# 06 — Sharpe Diagnosis (V2: BTC/USD)

Systematic debugging tool for when out-of-sample Sharpe is below the **0.5 target**.

Run these 6 diagnostic cells in order to isolate the failure mode:

| Cell | Diagnostic |
|------|-----------|
| 1 | Data integrity + feature normalization |
| 2 | Reward signal health |
| 3 | Action distribution (target 20–50% LONG) |
| 4 | Overfitting check (train vs val vs test Sharpe) |
| 5 | Transaction cost ablation (set λ=0) |
| 6 | Feature importance ablation (macro-only vs micro-only vs both) |

**Prerequisites:** Run notebooks 01–03 first. Load `BTC_USD.parquet` and `BTC_USD.pth`.

In [ ]:
!pip install -q torch torchvision gymnasium numpy pandas pyarrow pytz tqdm matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'

REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)
print('Repo on path ✓')

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from colab.deepscalper.utils import compute_macro_features, compute_micro_features, compute_day_starts
from colab.deepscalper.environment import ScalperEnv
from colab.deepscalper.agent import DeepScalperAgent

PAIR      = 'BTC/USD'
SAFE_NAME = 'BTC_USD'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
LOOKBACK  = 10

# Load data
bars = pd.read_parquet(f'{RAW_DIR}/{SAFE_NAME}.parquet')
bars.columns = [c.lower() for c in bars.columns]
bars = bars[['open','high','low','close','volume']].astype(float)

macro_feats = compute_macro_features(bars)
lob_feats   = compute_micro_features(bars)        # (n, 4)
close_arr   = bars['close'].values.astype(np.float64)
day_starts  = compute_day_starts(bars.index)

n = len(bars)
train_end = int(n * 0.70)
val_end   = int(n * 0.80)

# Split-aligned day starts
train_day_starts_all = [d for d in day_starts if d < train_end]
val_day_starts_all = [d - train_end for d in day_starts if train_end <= d < val_end]
test_day_starts_all = [d - val_end for d in day_starts if d >= val_end]

# Use identical fixed episode/day sets across all diagnostic cells.
# We cap to the smallest split so Train/Val/Test are directly comparable.
DIAG_DAY_COUNT = min(len(train_day_starts_all), len(val_day_starts_all), len(test_day_starts_all))
if DIAG_DAY_COUNT == 0:
    raise ValueError('No complete diagnostic day sets found across Train/Val/Test splits.')

DIAG_DAY_IDS = list(range(DIAG_DAY_COUNT))
TRAIN_DIAG_DAY_STARTS = [train_day_starts_all[i] for i in DIAG_DAY_IDS]
VAL_DIAG_DAY_STARTS = [val_day_starts_all[i] for i in DIAG_DAY_IDS]
TEST_DIAG_DAY_STARTS = [test_day_starts_all[i] for i in DIAG_DAY_IDS]

print(f'Dataset: {n:,} bars | Train: {train_end:,} | Val: {val_end-train_end:,} | Test: {n-val_end:,}')
print(f'Diagnostic day sets: {DIAG_DAY_COUNT} per split (fixed indices: {DIAG_DAY_IDS[0]}..{DIAG_DAY_IDS[-1]})')

## Cell 1 — Data Integrity + Feature Normalization

In [ ]:
# ── Cell 1: Data integrity + feature normalization ────────────────────────────
print('=== DATA INTEGRITY ===')
print(f'Close prices — min: {close_arr.min():.2f}, max: {close_arr.max():.2f}')
print(f'Zero/negative closes: {(close_arr <= 0).sum()}')
print(f'NaN in close_arr:     {np.isnan(close_arr).sum()}')

# Check time gaps > 2 minutes
time_diffs = bars.index.to_series().diff().dt.total_seconds().div(60)
large_gaps = time_diffs[time_diffs > 2]
print(f'Gaps > 2 min:         {len(large_gaps)}')
if len(large_gaps) > 0:
    print(f'  Largest gap: {large_gaps.max():.1f} min at {large_gaps.idxmax()}')

print()
print('=== MACRO FEATURE NORMALIZATION (train split) ===')
for i in range(macro_feats.shape[1]):
    col = macro_feats[:train_end, i]
    print(f'  feat[{i:2d}]: mean={col.mean():.3f}  std={col.std():.3f}  '
          f'min={col.min():.3f}  max={col.max():.3f}')

print()
print('=== LOB FEATURE NORMALIZATION (train split) ===')
lob_names = ['spread_pct', 'order_imbalance', 'depth_ratio', 'mid_move_1min']
for i, name in enumerate(lob_names):
    col = lob_feats[:train_end, i]
    print(f'  {name:20s}: mean={col.mean():.4f}  std={col.std():.4f}  '
          f'min={col.min():.4f}  max={col.max():.4f}')

# Flag abnormal feature ranges (should be approximately z-scored)
macro_std_mean = macro_feats[:train_end].std(axis=0).mean()
print(f'\nMacro feature avg std (should be ~1 for z-scored): {macro_std_mean:.3f}')
if macro_std_mean > 5 or macro_std_mean < 0.1:
    print('  ⚠️  Abnormal feature scale — check compute_macro_features normalization.')
else:
    print('  ✅ Feature scale OK')

## Cell 2 — Reward Signal Health

In [ ]:
# ── Cell 2: Reward signal health ─────────────────────────────────────────────
# Simulate the reward distribution from a random agent to establish the baseline
# Uses the shared fixed diagnostic day set.

train_env = ScalperEnv(
    lob_features      = lob_feats[:train_end],
    macro_features    = macro_feats[:train_end],
    close_prices      = close_arr[:train_end],
    day_starts        = TRAIN_DIAG_DAY_STARTS,
    lookback_bars     = LOOKBACK,
    random_day_reset  = False,
)

random_rewards = []
for _ in range(len(train_env.day_starts)):
    obs, _ = train_env.reset()
    done = False
    while not done:
        action = train_env.action_space.sample()
        obs, r, term, trunc, info = train_env.step(action)
        random_rewards.append(r)
        done = term or trunc

rr = np.array(random_rewards)
print('=== REWARD SIGNAL HEALTH (random policy, train env) ===')
print(f'  Day set: fixed shared diagnostic subset (N={len(train_env.day_starts)})')
print(f'  N samples:   {len(rr):,}')
print(f'  Mean reward: {rr.mean():.6f}  (expect small negative due to transaction costs)')
print(f'  Std reward:  {rr.std():.6f}')
print(f'  Min reward:  {rr.min():.6f}')
print(f'  Max reward:  {rr.max():.6f}')
print(f'  % positive:  {(rr > 0).mean()*100:.1f}% (random policy should be ~50%)')

if abs(rr.mean()) > 0.1:
    print('  ⚠️  Large mean reward magnitude — reward may not be well-scaled.')
    print('      Consider reducing TRANSACTION_COST_LAMBDA or HINDSIGHT_WEIGHT.')
else:
    print('  ✅ Reward scale looks healthy')

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(rr, bins=100, edgecolor='none', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='zero')
ax.set_xlabel('Reward')
ax.set_title('Reward Distribution (Random Policy, Train Env)')
ax.legend()
plt.tight_layout()
plt.show()

## Cell 3 — Action Distribution

In [ ]:
# ── Cell 3: Action distribution (target 20–50% LONG actions) ─────────────────
MACRO_DIM = 11; LOB_DIM = 4; PRIV_DIM = 2
N_DIR = 2; N_SIZE = 1; GRU_HIDDEN = 128; MACRO_EMBED = 64; FC_HIDDEN = 128

weights_path = f'{WEIGHTS_DIR}/{SAFE_NAME}.pth'
if not os.path.exists(weights_path):
    print(f'⚠️  Weights not found at {weights_path} — run 03_train first.')
else:
    agent = DeepScalperAgent(
        macro_dim=MACRO_DIM, lob_dim=LOB_DIM, priv_dim=PRIV_DIM,
        n_dir=N_DIR, n_size=N_SIZE, gru_hidden=GRU_HIDDEN,
        macro_embed=MACRO_EMBED, fc_hidden=FC_HIDDEN, device=DEVICE,
    )
    ckpt = torch.load(weights_path, map_location=DEVICE, weights_only=True)
    agent.online_net.load_state_dict(ckpt.get('online_net', ckpt))
    agent.epsilon = 0.0   # greedy

    test_env = ScalperEnv(
        lob_features      = lob_feats[val_end:],
        macro_features    = macro_feats[val_end:],
        close_prices      = close_arr[val_end:],
        day_starts        = TEST_DIAG_DAY_STARTS,
        lookback_bars     = LOOKBACK,
        random_day_reset  = False,
    )

    actions_taken = []
    for _ in range(len(test_env.day_starts)):
        obs, _ = test_env.reset()
        done = False
        while not done:
            dir_act, _ = agent.select_action(obs)
            obs, r, term, trunc, _ = test_env.step(dir_act)
            actions_taken.append(dir_act)
            done = term or trunc

    actions = np.array(actions_taken)
    long_pct = (actions == 1).mean() * 100
    flat_pct = (actions == 0).mean() * 100

    print('=== ACTION DISTRIBUTION (greedy policy, test set) ===')
    print(f'  Day set: fixed shared diagnostic subset (N={len(test_env.day_starts)})')
    print(f'  FLAT (0): {flat_pct:.1f}%')
    print(f'  LONG (1): {long_pct:.1f}%')

    if long_pct < 20:
        print('  ⚠️  Very few LONG actions — model may be over-penalized for trading.')
        print('      Try: reducing TRANSACTION_COST_LAMBDA, increasing HINDSIGHT_WEIGHT.')
    elif long_pct > 80:
        print('  ⚠️  Always LONG — model may not be learning FLAT conditions.')
        print('      Try: increasing risk_penalty coefficient in _compute_reward().')
    else:
        print(f'  ✅ Action mix looks healthy (target: 20–50% LONG)')

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['FLAT', 'LONG'], [flat_pct, long_pct], color=['steelblue', 'darkorange'])
    ax.axhline(20, linestyle='--', color='gray', label='20% floor')
    ax.axhline(80, linestyle='--', color='red',  label='80% ceiling')
    ax.set_ylabel('% of steps')
    ax.set_title('Action Distribution (Test Set)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Cell 4 — Overfitting Check (Train vs Val vs Test Sharpe)

In [ ]:
# ── Cell 4: Overfitting check ─────────────────────────────────────────────────
_CRYPTO_ANNUALISE = np.sqrt(525_960)

def compute_episode_sharpe(log_returns):
    rets = np.asarray(log_returns, dtype=np.float64)
    if len(rets) < 5 or rets.std() == 0:
        return 0.0
    return float((rets.mean() / rets.std()) * _CRYPTO_ANNUALISE)

if 'agent' not in dir():
    print('⚠️  Run Cell 3 first to load the agent.')
else:
    splits = {
        'Train': ScalperEnv(
            lob_features      = lob_feats[:train_end],
            macro_features    = macro_feats[:train_end],
            close_prices      = close_arr[:train_end],
            day_starts        = TRAIN_DIAG_DAY_STARTS,
            lookback_bars     = LOOKBACK,
            random_day_reset  = False,
        ),
        'Val': ScalperEnv(
            lob_features      = lob_feats[train_end:val_end],
            macro_features    = macro_feats[train_end:val_end],
            close_prices      = close_arr[train_end:val_end],
            day_starts        = VAL_DIAG_DAY_STARTS,
            lookback_bars     = LOOKBACK,
            random_day_reset  = False,
        ),
        'Test': ScalperEnv(
            lob_features      = lob_feats[val_end:],
            macro_features    = macro_feats[val_end:],
            close_prices      = close_arr[val_end:],
            day_starts        = TEST_DIAG_DAY_STARTS,
            lookback_bars     = LOOKBACK,
            random_day_reset  = False,
        ),
    }

    print('=== OVERFITTING CHECK (annualised Sharpe per split) ===')
    print(f'  Day set: fixed shared diagnostic subset (N={DIAG_DAY_COUNT} per split)')
    sharpes = {}
    for split_name, env in splits.items():
        all_returns = []
        for _ in range(len(env.day_starts)):
            obs, _ = env.reset()
            done = False
            while not done:
                dir_act, _ = agent.select_action(obs)
                obs, r, term, trunc, info = env.step(dir_act)
                all_returns.append(info.get('log_return', 0.0))
                done = term or trunc
        sharpe = compute_episode_sharpe(all_returns)
        sharpes[split_name] = sharpe
        print(f'  {split_name:5s}: {sharpe:.4f}')

    gap = sharpes.get('Train', 0) - sharpes.get('Test', 0)
    print(f'\n  Train-Test Sharpe gap: {gap:.4f}')
    if gap > 0.5:
        print('  ⚠️  Large overfitting gap. Try: more dropout, larger PER buffer, fewer episodes.')
    elif sharpes.get('Test', 0) < 0.5:
        print('  ⚠️  Test Sharpe below target 0.5 — model not generalizing well.')
        print('      Proceed to Cell 5 (cost ablation) and Cell 6 (feature ablation).')
    else:
        print('  ✅ Sharpe gap acceptable and test target met.')

    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['steelblue', 'darkorange', 'green' if sharpes.get('Test', 0) >= 0.5 else 'red']
    ax.bar(list(sharpes.keys()), list(sharpes.values()), color=colors)
    ax.axhline(0.5, linestyle='--', color='red', label='Target Sharpe 0.5')
    ax.set_ylabel('Annualised Sharpe')
    ax.set_title('Sharpe by Data Split')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Cell 5 — Transaction Cost Ablation (λ=0)

In [ ]:
# ── Cell 5: Transaction cost ablation ────────────────────────────────────────
# If Sharpe improves dramatically with λ=0, the real cost is destroying alpha.
# Solutions: (a) trade less frequently, (b) seek higher-return signals.

if 'agent' not in dir():
    print('⚠️  Run Cell 3 first to load the agent.')
else:
    for cost in [0.0025, 0.0]:
        env_zero_cost = ScalperEnv(
            lob_features         = lob_feats[val_end:],
            macro_features       = macro_feats[val_end:],
            close_prices         = close_arr[val_end:],
            day_starts           = TEST_DIAG_DAY_STARTS,
            lookback_bars        = LOOKBACK,
            transaction_cost_pct = cost,
            random_day_reset     = False,
        )
        all_returns = []
        for _ in range(len(env_zero_cost.day_starts)):
            obs, _ = env_zero_cost.reset()
            done = False
            while not done:
                dir_act, _ = agent.select_action(obs)
                obs, r, term, trunc, info = env_zero_cost.step(dir_act)
                all_returns.append(info.get('log_return', 0.0))
                done = term or trunc
        sharpe = compute_episode_sharpe(all_returns)
        label = f'λ={cost:.4f}'
        print(f'  Test Sharpe ({label}): {sharpe:.4f}')

    print()
    print('  Interpretation:')
    print('  - If λ=0 Sharpe >> λ=0.0025: costs are the primary drag → trade less often.')
    print('  - If λ=0 Sharpe still low:   signal quality is the problem → more training data.')

## Cell 6 — Feature Importance Ablation

In [ ]:
# ── Cell 6: Feature importance ablation ──────────────────────────────────────
# Test: macro features only vs. LOB features only vs. both (baseline)
# This reveals whether the model relies on macro or micro signals.

if 'agent' not in dir():
    print('⚠️  Run Cell 3 first to load the agent.')
else:
    def eval_ablation(lob_override, macro_override, label):
        env_abl = ScalperEnv(
            lob_features      = lob_override,
            macro_features    = macro_override,
            close_prices      = close_arr[val_end:],
            day_starts        = TEST_DIAG_DAY_STARTS,
            lookback_bars     = LOOKBACK,
            random_day_reset  = False,
        )
        all_returns = []
        for _ in range(len(env_abl.day_starts)):
            obs, _ = env_abl.reset()
            done = False
            while not done:
                dir_act, _ = agent.select_action(obs)
                obs, r, term, trunc, info = env_abl.step(dir_act)
                all_returns.append(info.get('log_return', 0.0))
                done = term or trunc
        sharpe = compute_episode_sharpe(all_returns)
        print(f'  {label:35s}: Sharpe = {sharpe:.4f}')
        return sharpe

    n_test = len(close_arr) - val_end
    zeros_lob   = np.zeros((n_test, lob_feats.shape[1]),   dtype=np.float32)
    zeros_macro = np.zeros((n_test, macro_feats.shape[1]), dtype=np.float32)

    print('=== FEATURE IMPORTANCE ABLATION (test set) ===')
    print(f'  Day set: fixed shared diagnostic subset (N={DIAG_DAY_COUNT})')
    s_both  = eval_ablation(lob_feats[val_end:], macro_feats[val_end:], 'Both (baseline)')
    s_macro = eval_ablation(zeros_lob,           macro_feats[val_end:], 'Macro only (LOB zeroed)')
    s_lob   = eval_ablation(lob_feats[val_end:], zeros_macro,           'LOB only (macro zeroed)')

    print()
    print('  Interpretation:')
    if s_macro > s_lob:
        print('  - Macro features carry more signal. LOB proxy may need improvement.')
        print('    Solution: collect more LOB data (07_lob_recorder.ipynb) for v3.')
    elif s_lob > s_macro:
        print('  - LOB features carry more signal. Macro features may be noisy.')
        print('    Solution: review compute_macro_features normalization (Cell 1).')
    else:
        print('  - Both feature sets contribute roughly equally.')

    fig, ax = plt.subplots(figsize=(7, 3))
    labels_ = ['Both', 'Macro only', 'LOB only']
    vals_   = [s_both, s_macro, s_lob]
    colors_ = ['green' if v >= 0.5 else 'steelblue' for v in vals_]
    ax.bar(labels_, vals_, color=colors_)
    ax.axhline(0.5, linestyle='--', color='red', label='Target 0.5')
    ax.set_ylabel('Annualised Sharpe')
    ax.set_title('Feature Ablation (Test Set)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Acceptance Gates: Risk-Adjusted Return
Use this gate before promoting weights to live paper trading.

The policy is accepted only if all checks pass:
- Sharpe greater than random-policy baseline on the same fixed day set
- Max drawdown under the configured limit
- Net expectancy per trade (after fees) greater than zero

In [ ]:
# -- Acceptance gates: Sharpe vs baseline, drawdown cap, expectancy after fees --
if 'agent' not in dir():
    print('WARNING: Run Cell 6 first to load the trained agent.')
else:
    GATE_MAX_DRAWDOWN_PCT = 15.0
    GATE_EXPECTANCY_MIN = 0.0
    GATE_SHARPE_MARGIN = 0.0  # require model Sharpe > baseline + margin

    gate_env = ScalperEnv(
        lob_features=lob_feats[val_end:],
        macro_features=macro_feats[val_end:],
        close_prices=close_arr[val_end:],
        day_starts=TEST_DIAG_DAY_STARTS,
        lookback_bars=LOOKBACK,
        random_day_reset=False,
    )

    def eval_policy_metrics(env, use_random_policy: bool = False):
        net_step_returns = []
        entries = 0
        for _ in range(len(env.day_starts)):
            obs, _ = env.reset()
            done = False
            prev_pos = int(obs['priv'][-1, 0])
            while not done:
                if use_random_policy:
                    dir_act = int(env.action_space.sample())
                else:
                    dir_act, _ = agent.select_action(obs)
                next_obs, _r, term, trunc, info = env.step(dir_act)
                done = term or trunc

                pos = int(info.get('position', prev_pos))
                trade_occurred = int(pos != prev_pos)
                entry = int(prev_pos == 0 and pos == 1)
                entries += entry

                net_ret = float(info.get('log_return', 0.0)) - trade_occurred * float(env.transaction_cost_pct)
                net_step_returns.append(net_ret)

                obs = next_obs
                prev_pos = pos

        rets = np.asarray(net_step_returns, dtype=np.float64)
        if len(rets) == 0:
            return {
                'sharpe': 0.0,
                'max_dd_pct': 0.0,
                'expectancy_per_trade': -np.inf,
                'entries': 0,
                'n_steps': 0,
            }

        sharpe = compute_episode_sharpe(rets.tolist())
        curve = np.cumsum(rets)
        peak = np.maximum.accumulate(curve)
        max_dd = float(np.max(peak - curve))
        max_dd_pct = max_dd * 100.0
        expectancy_per_trade = float(rets.sum() / entries) if entries > 0 else -np.inf

        return {
            'sharpe': sharpe,
            'max_dd_pct': max_dd_pct,
            'expectancy_per_trade': expectancy_per_trade,
            'entries': entries,
            'n_steps': len(rets),
        }

    model_m = eval_policy_metrics(gate_env, use_random_policy=False)
    baseline_m = eval_policy_metrics(gate_env, use_random_policy=True)

    gate_sharpe = model_m['sharpe'] > (baseline_m['sharpe'] + GATE_SHARPE_MARGIN)
    gate_drawdown = model_m['max_dd_pct'] <= GATE_MAX_DRAWDOWN_PCT
    gate_expectancy = model_m['expectancy_per_trade'] > GATE_EXPECTANCY_MIN
    gate_all = gate_sharpe and gate_drawdown and gate_expectancy

    print('=== ACCEPTANCE GATES (RISK-ADJUSTED) ===')
    print(f"Day set: fixed shared diagnostic subset (N={len(gate_env.day_starts)})")
    print('')
    print(f"Model Sharpe (net after fees):    {model_m['sharpe']:.4f}")
    print(f"Baseline Sharpe (random policy):  {baseline_m['sharpe']:.4f}")
    print(f"Max Drawdown (%):                 {model_m['max_dd_pct']:.2f}")
    print(f"Expectancy per trade (net):       {model_m['expectancy_per_trade']:.6f}")
    print(f"Trades (entries):                 {model_m['entries']}")
    print('')

    print(f"[{'OK' if gate_sharpe else 'FAIL'}] Sharpe > baseline + margin ({GATE_SHARPE_MARGIN:.2f})")
    print(f"[{'OK' if gate_drawdown else 'FAIL'}] Max drawdown <= {GATE_MAX_DRAWDOWN_PCT:.1f}%")
    print(f"[{'OK' if gate_expectancy else 'FAIL'}] Expectancy per trade > {GATE_EXPECTANCY_MIN:.4f}")
    print('')
    if gate_all:
        print('[ACCEPTED] Model passes risk-adjusted acceptance gates.')
    else:
        print('[REJECTED] Model fails one or more acceptance gates.')
        print('Next step: tune reward/cost tradeoff and rerun 03 then this diagnosis notebook.')